# Hint Generator Testing Notebook

Testing notebook for the `hint_generator` chain which provides progressive hints for coding tasks.

**Chain Functions:**
- `generate_hint(task, student_attempt, level)` - Single hint at specified level
- `get_all_hints(task, student_attempt)` - All 3 hint levels

## 1. Setup

In [ ]:
import os
import sys
from dotenv import load_dotenv

# Add parent directory to path for imports
sys.path.insert(0, os.path.abspath('../..'))

load_dotenv()
print("Environment loaded")

In [ ]:
from ai_chains.chains import hint_generator

print("Modules imported")

## 2. Chain Configuration

In [ ]:
# Display prompt template
with open('../../ai_chains/prompts/hint_generator.yaml', 'r', encoding='utf-8') as f:
    print("Prompt Template:")
    print(f.read())

In [ ]:
# Show hint level definitions
print("Hint Levels:")
print("  Level 1 (Halus): Gentle direction, points to the right path")
print("  Level 2 (Konseptual): Concept explanation with similar example")
print("  Level 3 (Langsung): Direct approach with missing parts shown")

In [ ]:
# Show LLM configuration
print("LLM Configuration:")
print(f"  OpenRouter Model: {os.getenv('OPENROUTER_MODEL', 'google/gemma-7b-it')}")
print(f"  Z.AI Model: {os.getenv('ZAI_MODEL', 'glm-4.7')}")
print(f"  OpenRouter Key: {'Set' if os.getenv('OPENROUTER_API_KEY') else 'Not set'}")
print(f"  Z.AI Key: {'Set' if os.getenv('ZAI_API_KEY') else 'Not set'}")

print("Test 5: Real Course Task")
print("="*50)

# Task from session_1_1
task = "Buat variabel `nama` dengan nama Anda, variabel `umur` dengan angka, dan cetak semua"
attempt = """
nama = 'Budi'
umur = '25'
print(nama, umur)
"""

print(f"\\nTask: {task}")
print(f"Attempt:\\n{attempt.strip()}")

print("\\n--- Progressive Hints ---")
for level in range(1, 4):
    try:
        result = hint_generator.generate_hint(task, attempt, level)
        print(f"\\nLevel {level}: {result[:150]}...")
    except Exception as e:
        print(f"\\nLevel {level}: Error - {e}")

### Test 1: Level 1 - Gentle Hint

In [ ]:
print("Test 1: Level 1 (Halus) - Gentle Hint")
print("="*50)

task = "Buat variabel 'nama' dengan nilai nama Anda"
attempt = ""  # Student hasn't tried yet

result = hint_generator.generate_hint(task, attempt, level=1)

print(f"\nTask: {task}")
print(f"Attempt: (empty)")
print(f"\nLevel 1 Hint:\n{result}")

### Test 2: Level 2 - Conceptual Hint

In [ ]:
print("Test 2: Level 2 (Konseptual) - Conceptual Hint")
print("="*50)

task = "Buat variabel 'nama' dengan nilai nama Anda"
attempt = "nama == 'Budi'"  # Wrong syntax

result = hint_generator.generate_hint(task, attempt, level=2)

print(f"\nTask: {task}")
print(f"Attempt: {attempt}")
print(f"\nLevel 2 Hint:\n{result}")

### Test 3: Level 3 - Direct Hint

In [ ]:
print("Test 3: Level 3 (Langsung) - Direct Hint")
print("="*50)

task = "Buat variabel 'nama' dengan nilai nama Anda"
attempt = "var nama = 'Budi'"  # Wrong keyword

result = hint_generator.generate_hint(task, attempt, level=3)

print(f"\nTask: {task}")
print(f"Attempt: {attempt}")
print(f"\nLevel 3 Hint:\n{result}")

### Test 4: Get All Hints at Once

In [ ]:
print("Test 4: All Hints at Once")
print("="*50)

task = "Buat program yang mencetak 'Hello World'"
attempt = "print Hello World"  # Missing parentheses

hints = hint_generator.get_all_hints(task, attempt)

print(f"\nTask: {task}")
print(f"Attempt: {attempt}")

level_names = ["Halus", "Konseptual", "Langsung"]
for i, (name, hint) in enumerate(zip(level_names, hints), 1):
    print(f"\nLevel {i} ({name}):\n{hint}")

### Test 5: Real Course Task

In [ ]:
print("Test 5: Real Course Task")
print("="*50)

# Task from session_1_1
task = "Buat variabel `nama` dengan nama Anda, variabel `umur` dengan angka, dan cetak semua"
attempt = """
nama = 'Budi'
umur = '25'
print(nama, umur)
"""

print(f"\nTask: {task}")
print(f"Attempt:\n{attempt.strip()}")

print("\n--- Progressive Hints ---")
for level in range(1, 4):
    result = hint_generator.generate_hint(task, attempt, level)
    print(f"\nLevel {level}: {result[:150]}...")

### Test 6: Different Task Types

In [ ]:
print("Test 6: Different Task Types")
print("="*50)

tasks = [
    {
        "task": "Konversi string '100' menjadi integer",
        "attempt": "angka = '100'"
    },
    {
        "task": "Ubah 'hello' menjadi 'HELLO'",
        "attempt": "text.upper"  # Missing ()
    },
    {
        "task": "Cetak tanpa baris baru",
        "attempt": "print('Loading')"
    }
]

for i, t in enumerate(tasks, 1):
    hint = hint_generator.generate_hint(t["task"], t["attempt"], level=1)
    print(f"\n{i}. Task: {t['task']}")
    print(f"   Attempt: {t['attempt']}")
    print(f"   Hint: {hint[:120]}...")

## 4. Analysis

In [ ]:
print("Analysis Summary")
print("="*50)

# Test progressive hint quality
task = "Buat variabel untuk menyimpan umur"
attempt = ""

print("\nProgressive Hint Quality Check:")
hints = hint_generator.get_all_hints(task, attempt)

# Check that hints progress in specificity
for i, hint in enumerate(hints, 1):
    # Basic quality checks
    has_indo = any(word in hint.lower() for word in ["gunakan", "coba", " variabel", "python"])
    has_content = len(hint) > 20
    
    status = "OK" if (has_indo and has_content) else "CHECK"
    print(f"  [{status}] Level {i}: Length={len(hint)}, Indonesian content={has_indo}")

# Check that level 3 is more direct than level 1
print("\nProgression Check:")
print(f"  Level 1 length: {len(hints[0])}")
print(f"  Level 2 length: {len(hints[1])}")
print(f"  Level 3 length: {len(hints[2])}")

## 5. Notes

**Hint Levels:**
- **Level 1 (Halus)**: Points in the right direction without giving away
- **Level 2 (Konseptual)**: Explains the concept with similar examples
- **Level 3 (Langsung)**: Shows direct approach with partial solution

**Expected Output:**
- All hints in Indonesian
- Progressive specificity from level 1 to 3
- Encouraging tone
- References the student's attempt if provided

**Fallback Behavior:**
- Returns: `"Hint tidak tersedia. Error: {error}"`